In [1]:
!pip install -q langchain-huggingface langchain-chroma langchain-core langchain-text-splitters langchain-community langchain-classic langchain-groq sentence-transformers transformers chromadb


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 124.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 75.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/9

In [ ]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

# Retrieval

In [3]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from google.colab import drive
drive.mount("/content/drive")

persist_directory = "/content/drive/MyDrive/law_chatbot_chroma"
embedding = HuggingFaceEmbeddings(model_name="BAAI/bge-m3", model_kwargs={"device": "cuda"})
vectordb = Chroma(persist_directory=persist_directory, embedding_function=embedding)
print(vectordb._collection.count())


Mounted at /content/drive


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

25286


## Addressing diversity: Maximum Marginal Relevance (MMR)

In [4]:
question = "ما هي شروط فسخ عقد الايجار؟"
docs_ss = vectordb.similarity_search(question, k=3)
docs_mmr = vectordb.max_marginal_relevance_search(question, k=3)

print("--- plain similarity search ---")
for d in docs_ss:
    print(d.page_content[:100], "\n")

print("--- MMR ---")
for d in docs_mmr:
    print(d.page_content[:100], "\n")

--- plain similarity search ---
جلسة 21/3/1999 برئاسة الشيخ خليفة بن محمد ال خليفة وعضوية السادة المستشارين علي يوسف منصور وكيل المح 

جلسة 10 من ديسمبر سنة 2007 برئاسة الشيخ / خليفة بن راشد بن عبدالله ال خليفة رئيس المحكمة وعضوية المس 

جلسة 19 من ابريل سنة 2010 برئاسة الشيخ خليفة بن راشد بن عبدالله ال خليفة رئيس المحكمة وعضوية المستشا 

--- MMR ---
جلسة 21/3/1999 برئاسة الشيخ خليفة بن محمد ال خليفة وعضوية السادة المستشارين علي يوسف منصور وكيل المح 

___________________________________________________________ جلسة 13 من مايو سنة 2013 برئاسة الشيخ /  

مادة (60) معاملة عقود الايجار القائمة يلتزم امين التفليسة بسداد اجرة عقود الايجار في موعد استحقاقها، 



## Addressing specificity: metadata filtering

In [6]:
docs = vectordb.similarity_search(
    question,
    k=3,
    filter={"source": "lloc"},  # legislation only
)
for d in docs:
    print(d.metadata)

{'doc_id': 'K2218', 'title': 'قانون رقم (22) لسنة 2018 بإصدار قانون إعادة التنظيم والإفلاس', 'article_no': '60', 'source': 'lloc', 'categories': 'التشريعات المالية والاقتصادية, تشريعات المرأة والطفل'}
{'title': 'قانون رقم (27) لسنة 2014 بإصدار قانون إيجار العقارات', 'source': 'lloc', 'doc_id': 'K2714', 'categories': 'التشريعات المالية والاقتصادية, تشريعات الإسكان والبناء والتخطيط العمراني والعقاري', 'article_no': '38'}
{'doc_id': 'K2714', 'source': 'lloc', 'article_no': '36', 'title': 'قانون رقم (27) لسنة 2014 بإصدار قانون إيجار العقارات', 'categories': 'التشريعات المالية والاقتصادية, تشريعات الإسكان والبناء والتخطيط العمراني والعقاري'}


## Self-query retriever

In [8]:
!pip install -q -U langchain-community langchain-classic

In [20]:
from langchain_groq import ChatGroq
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_classic.retrievers.self_query.chroma import ChromaTranslator
from langchain_classic.chains.query_constructor.base import AttributeInfo

metadata_field_info = [
    AttributeInfo(
        name="source",
        description="مصدر هذا المقطع، احد ثلاثة:\nlloc: التشريعات والقوانين\nsjc: احكام محكمة التمييز\nccb: احكام المحكمة الدستورية",
        type="string",
    ),
    AttributeInfo(
        name="doc_id",
        description="رمز القانون او رقم القضية والطعن الذي يعود اليه هذا المقطع",
        type="string",
    ),
]

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

document_content_description = "نصوص قانونية بحرينية، مواد تشريعية واحكام قضائية"
retriever = SelfQueryRetriever.from_llm(
    llm,
    vectordb,
    document_content_description,
    metadata_field_info,
    verbose=True,
    structured_query_translator=ChromaTranslator(),
)

In [21]:
docs = retriever.invoke(question)
for d in docs:
    print(d.metadata)

{'case_type': 'مدني', 'doc_id': '254 M 2007 K 281', 'source': 'sjc'}
{'doc_id': 'K2714', 'source': 'lloc', 'categories': 'التشريعات المالية والاقتصادية, تشريعات الإسكان والبناء والتخطيط العمراني والعقاري', 'article_no': '13', 'title': 'قانون رقم (27) لسنة 2014 بإصدار قانون إيجار العقارات'}
{'source': 'sjc', 'doc_id': '256 M 2009 K 83', 'case_type': 'مدني'}
{'doc_id': '100 M 1996 K 141', 'case_type': 'مدني', 'source': 'sjc'}


In [22]:
import os
key = os.environ.get("GROQ_API_KEY")
print("Key loaded:", bool(key))
print("Length:", len(key) if key else 0)
print("Starts with gsk_:", key.startswith("gsk_") if key else False)

Key loaded: True
Length: 56
Starts with gsk_: True


## Other retrieval types

In [25]:
from langchain_community.retrievers import TFIDFRetriever

all_texts = vectordb.get()["documents"]
tfidf_retriever = TFIDFRetriever.from_texts(all_texts)

docs_tfidf = tfidf_retriever.invoke(question)
docs_tfidf[0].page_content[:200] if docs_tfidf else "no results"

'جلسة 2 من يونيه سنة 2008 برئاسة الشيخ / خليفة بن راشد بن عبدالله ال خليفة رئيس المحكمة وعضوية المستشارين / مسعد رمضان الساعي, محسن محمد فضلي و محي الدين السيد حسن. (146) الطعن رقم 346 لسنة 2007 (1-3) '